In [1]:
import numpy as np

from PIL import Image
from pathlib import Path
from collections import Counter
from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score



In [2]:


# Lokasi dataset
DATASET_DIR = Path("../dataset")

TRAIN_DIR = DATASET_DIR / "Train_Set"
TEST_DIR = DATASET_DIR / "Test_Set"

print("Train path :", TRAIN_DIR)
print("Test path  :", TEST_DIR)

print("\nTrain exists:", TRAIN_DIR.exists())
print("Test exists :", TEST_DIR.exists())

Train path : ..\dataset\Train_Set
Test path  : ..\dataset\Test_Set

Train exists: True
Test exists : True


In [3]:
IMG_EXTS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff", ".jfif"
}

for class_name in ["Aman", "Lainnya", "Rusak"]:
    folder = TRAIN_DIR / class_name

    count = sum(
        1 for p in folder.iterdir()
        if p.suffix.lower() in IMG_EXTS
    )

    print(f"{class_name}: {count} gambar")

Aman: 8800 gambar
Lainnya: 423 gambar
Rusak: 3001 gambar


In [4]:
test_files = [
    p for p in TEST_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMG_EXTS
]

print("Jumlah gambar Test:", len(test_files))

Jumlah gambar Test: 3057


In [5]:

def get_image_info(files):
    sizes = []
    formats = []

    for path in files:
        try:
            with Image.open(path) as img:
                sizes.append(img.size)
                formats.append(img.format)
        except Exception:
            pass

    return sizes, formats

train_files = []

for class_name in ["Aman", "Lainnya", "Rusak"]:
    folder = TRAIN_DIR / class_name

    train_files.extend([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ])

train_sizes, train_formats = get_image_info(train_files)
test_sizes, test_formats = get_image_info(test_files)


print("TRAIN")
print("Jumlah berhasil dibaca :", len(train_sizes))
print("Format :", Counter(train_formats))
print("Ukuran paling umum :", Counter(train_sizes).most_common(10))

print("\nTEST")
print("Jumlah berhasil dibaca :", len(test_sizes))
print("Format :", Counter(test_formats))
print("Ukuran paling umum :", Counter(test_sizes).most_common(10))

TRAIN
Jumlah berhasil dibaca : 12224
Format : Counter({'JPEG': 10757, 'WEBP': 1371, 'PNG': 95, 'AVIF': 1})
Ukuran paling umum : [((640, 640), 2785), ((512, 512), 2492), ((675, 900), 499), ((416, 416), 298), ((224, 224), 253), ((736, 736), 139), ((267, 267), 114), ((736, 981), 76), ((161, 251), 56), ((576, 1280), 55)]

TEST
Jumlah berhasil dibaca : 3057
Format : Counter({'JPEG': 2315, 'WEBP': 597, 'PNG': 145})
Ukuran paling umum : [((224, 224), 520), ((640, 640), 347), ((675, 900), 280), ((416, 416), 193), ((3000, 4000), 55), ((736, 736), 55), ((512, 512), 45), ((2816, 1536), 40), ((267, 267), 40), ((4000, 3000), 37)]


In [6]:


def dhash(path, hash_size=8):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((hash_size + 1, hash_size))

        pixels = np.array(img)

        diff = pixels[:, 1:] > pixels[:, :-1]

        # Ubah menjadi string 0 dan 1
        return "".join("1" if x else "0" for x in diff.flatten())


print("Contoh dHash:")
print(dhash(train_files[0]))

Contoh dHash:
1100010011100101001001011000110111100000111000101111001001100000


In [7]:
hash_groups = defaultdict(list)

for path in train_files:
    try:
        h = dhash(path)
        hash_groups[h].append(path)
    except Exception:
        pass

# Hanya ambil hash yang muncul lebih dari sekali
duplicates = {
    h: paths
    for h, paths in hash_groups.items()
    if len(paths) > 1
}

print("Jumlah kelompok duplicate:", len(duplicates))

duplicate_images = sum(len(paths) for paths in duplicates.values())

print("Total gambar yang termasuk duplicate:", duplicate_images)

d:\JamTerbang\datascape\.venv\Lib\site-packages\PIL\Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Jumlah kelompok duplicate: 871
Total gambar yang termasuk duplicate: 2018


In [8]:
conflict_groups = []

for h, paths in duplicates.items():

    labels = []

    for path in paths:
        label = path.parent.name
        labels.append(label)

    unique_labels = set(labels)

    if len(unique_labels) > 1:
        conflict_groups.append((h, paths, unique_labels))


print("Kelompok duplicate dengan label berbeda:",
      len(conflict_groups))

Kelompok duplicate dengan label berbeda: 46


In [9]:
conflict_pairs = Counter()

for h, paths, labels in conflict_groups:
    labels = sorted(labels)
    
    # Simpan kombinasi label
    pair = tuple(labels)
    conflict_pairs[pair] += 1

print("Kombinasi label yang konflik:")
for pair, count in conflict_pairs.items():
    print(f"{pair}: {count} kelompok")

Kombinasi label yang konflik:
('Aman', 'Rusak'): 45 kelompok
('Aman', 'Lainnya'): 1 kelompok


In [10]:
test_hash_groups = defaultdict(list)

for path in test_files:
    try:
        h = dhash(path)
        test_hash_groups[h].append(path)
    except Exception:
        pass

# Cari hash yang muncul baik di train maupun test
train_test_duplicates = []

for h, test_paths in test_hash_groups.items():
    if h in hash_groups:
        train_test_duplicates.append(
            (h, hash_groups[h], test_paths)
        )

print("Kelompok duplicate Train-Test:",
      len(train_test_duplicates))

print("Jumlah gambar Test yang punya duplicate di Train:",
      sum(len(test_paths) for _, _, test_paths in train_test_duplicates))

Kelompok duplicate Train-Test: 321
Jumlah gambar Test yang punya duplicate di Train: 424


In [11]:
# Tampilkan beberapa contoh pasangan Train-Test yang duplicate

for i, (h, train_paths, test_paths) in enumerate(train_test_duplicates[:10]):

    print(f"\n--- Duplicate {i+1} ---")

    print("TRAIN:")
    for path in train_paths:
        print(" ", path)

    print("TEST:")
    for path in test_paths:
        print(" ", path)


--- Duplicate 1 ---
TRAIN:
  ..\dataset\Train_Set\Aman\Aman_08249.jpg
  ..\dataset\Train_Set\Aman\Aman_08385.jpg
  ..\dataset\Train_Set\Aman\Aman_08548.jpg
TEST:
  ..\dataset\Test_Set\test_00002.jpg
  ..\dataset\Test_Set\test_00887.jpg

--- Duplicate 2 ---
TRAIN:
  ..\dataset\Train_Set\Rusak\Rusak_02752.jpg
TEST:
  ..\dataset\Test_Set\test_00003.jpg
  ..\dataset\Test_Set\test_01591.jpg
  ..\dataset\Test_Set\test_03057.jpg

--- Duplicate 3 ---
TRAIN:
  ..\dataset\Train_Set\Rusak\Rusak_01305.jpg
  ..\dataset\Train_Set\Rusak\Rusak_01306.jpg
  ..\dataset\Train_Set\Rusak\Rusak_01307.jpg
  ..\dataset\Train_Set\Rusak\Rusak_01308.jpg
TEST:
  ..\dataset\Test_Set\test_00006.jpg
  ..\dataset\Test_Set\test_01357.jpg
  ..\dataset\Test_Set\test_01462.jpg
  ..\dataset\Test_Set\test_01525.jpg
  ..\dataset\Test_Set\test_02265.jpg

--- Duplicate 4 ---
TRAIN:
  ..\dataset\Train_Set\Aman\Aman_08537.jpg
TEST:
  ..\dataset\Test_Set\test_00017.jpg
  ..\dataset\Test_Set\test_00552.jpg

--- Duplicate 5 ---
TR

In [12]:
correct = 0
wrong = 0
no_twin = 0

for h, paths in hash_groups.items():

    # Kalau hanya ada 1 gambar, tidak ada twin
    if len(paths) < 2:
        no_twin += 1
        continue

    # Ambil semua label dalam kelompok
    labels = [path.parent.name for path in paths]

    # Kalau semua label sama
    if len(set(labels)) == 1:
        correct += len(paths)

    # Kalau ada label berbeda
    else:
        wrong += len(paths)

total_with_twin = correct + wrong

print("Gambar dengan twin:", total_with_twin)
print("Twin dengan label konsisten:", correct)
print("Twin dengan label konflik:", wrong)

if total_with_twin > 0:
    accuracy = correct / total_with_twin
    print(f"Akurasi konsistensi twin: {accuracy:.4f}")

Gambar dengan twin: 2018
Twin dengan label konsisten: 1906
Twin dengan label konflik: 112
Akurasi konsistensi twin: 0.9445


In [13]:
conflict_image_count = 0

for h, paths, labels in conflict_groups:
    conflict_image_count += len(paths)

print("Total gambar dalam kelompok konflik:",
      conflict_image_count)

Total gambar dalam kelompok konflik: 112


In [14]:
conflict_label_count = Counter()

for h, paths, labels in conflict_groups:
    for path in paths:
        conflict_label_count[path.parent.name] += 1

print("\nDistribusi label dalam kelompok konflik:")

for label, count in conflict_label_count.items():
    print(f"{label}: {count}")


Distribusi label dalam kelompok konflik:
Aman: 54
Rusak: 57
Lainnya: 1


In [15]:
from collections import Counter

train_size_count = Counter(train_sizes)
test_size_count = Counter(test_sizes)

print("TRAIN - 10 ukuran paling sering:")
for size, count in train_size_count.most_common(10):
    percentage = count / len(train_sizes) * 100
    print(f"{size}: {count} ({percentage:.2f}%)")

print("\nTEST - 10 ukuran paling sering:")
for size, count in test_size_count.most_common(10):
    percentage = count / len(test_sizes) * 100
    print(f"{size}: {count} ({percentage:.2f}%)")

TRAIN - 10 ukuran paling sering:
(640, 640): 2785 (22.78%)
(512, 512): 2492 (20.39%)
(675, 900): 499 (4.08%)
(416, 416): 298 (2.44%)
(224, 224): 253 (2.07%)
(736, 736): 139 (1.14%)
(267, 267): 114 (0.93%)
(736, 981): 76 (0.62%)
(161, 251): 56 (0.46%)
(576, 1280): 55 (0.45%)

TEST - 10 ukuran paling sering:
(224, 224): 520 (17.01%)
(640, 640): 347 (11.35%)
(675, 900): 280 (9.16%)
(416, 416): 193 (6.31%)
(3000, 4000): 55 (1.80%)
(736, 736): 55 (1.80%)
(512, 512): 45 (1.47%)
(2816, 1536): 40 (1.31%)
(267, 267): 40 (1.31%)
(4000, 3000): 37 (1.21%)


In [16]:
def get_aspect_ratios(sizes):
    return [width / height for width, height in sizes]

train_ratios = get_aspect_ratios(train_sizes)
test_ratios = get_aspect_ratios(test_sizes)

def ratio_category(ratio):
    if 0.95 <= ratio <= 1.05:
        return "Square"
    elif ratio < 0.95:
        return "Portrait"
    else:
        return "Landscape"

train_ratio_categories = Counter(
    ratio_category(r) for r in train_ratios
)

test_ratio_categories = Counter(
    ratio_category(r) for r in test_ratios
)

print("TRAIN:")
for category, count in train_ratio_categories.items():
    percentage = count / len(train_ratios) * 100
    print(f"{category}: {count} ({percentage:.2f}%)")

print("\nTEST:")
for category, count in test_ratio_categories.items():
    percentage = count / len(test_ratios) * 100
    print(f"{category}: {count} ({percentage:.2f}%)")

TRAIN:
Landscape: 2449 (20.03%)
Square: 7776 (63.61%)
Portrait: 1999 (16.35%)

TEST:
Portrait: 676 (22.11%)
Square: 1435 (46.94%)
Landscape: 946 (30.95%)


In [17]:
def extract_features(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img = img.resize((32, 32))

        arr = np.array(img).astype(np.float32) / 255.0

    # Rata-rata warna RGB
    mean_rgb = arr.mean(axis=(0, 1))

    # Standar deviasi warna RGB
    std_rgb = arr.std(axis=(0, 1))

    # Aspect ratio
    width, height = Image.open(path).size
    aspect_ratio = width / height

    return np.concatenate([
        mean_rgb,
        std_rgb,
        [aspect_ratio]
    ])

In [18]:
print("Mengekstrak fitur Train...")

X_train_adv = np.array([
    extract_features(path)
    for path in train_files
])

print("Mengekstrak fitur Test...")

X_test_adv = np.array([
    extract_features(path)
    for path in test_files
])

print("Shape Train:", X_train_adv.shape)
print("Shape Test :", X_test_adv.shape)

Mengekstrak fitur Train...
Mengekstrak fitur Test...
Shape Train: (12224, 7)
Shape Test : (3057, 7)


In [19]:
# Gabungkan fitur Train dan Test
X_adv = np.vstack([X_train_adv, X_test_adv])

# Label:
# 0 = Train
# 1 = Test
y_adv = np.concatenate([
    np.zeros(len(X_train_adv)),
    np.ones(len(X_test_adv))
])

print("Shape X:", X_adv.shape)
print("Shape y:", y_adv.shape)

Shape X: (15281, 7)
Shape y: (15281,)


In [20]:
X_adv_train, X_adv_val, y_adv_train, y_adv_val = train_test_split(
    X_adv,
    y_adv,
    test_size=0.2,
    random_state=42,
    stratify=y_adv
)

model_adv = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_adv.fit(X_adv_train, y_adv_train)

prob_adv = model_adv.predict_proba(X_adv_val)[:, 1]

auc = roc_auc_score(y_adv_val, prob_adv)

print(f"Adversarial Validation AUC: {auc:.4f}")

Adversarial Validation AUC: 0.7317


In [21]:
feature_names = [
    "mean_R",
    "mean_G",
    "mean_B",
    "std_R",
    "std_G",
    "std_B",
    "aspect_ratio"
]

importance = model_adv.feature_importances_

for name, value in sorted(
    zip(feature_names, importance),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{name:15s}: {value:.4f}")

mean_R         : 0.1548
std_B          : 0.1527
std_R          : 0.1480
std_G          : 0.1461
mean_G         : 0.1458
mean_B         : 0.1444
aspect_ratio   : 0.1081


In [22]:
twin_test_consistent = 0
twin_test_conflict = 0

for h, train_paths, test_paths in train_test_duplicates:

    train_labels = [path.parent.name for path in train_paths]

    if len(set(train_labels)) == 1:
        twin_test_consistent += len(test_paths)
    else:
        twin_test_conflict += len(test_paths)

print("Test dengan twin dan label konsisten :", twin_test_consistent)
print("Test dengan twin tetapi label konflik:", twin_test_conflict)
print("Total Test dengan twin              :", 
      twin_test_consistent + twin_test_conflict)

Test dengan twin dan label konsisten : 412
Test dengan twin tetapi label konflik: 12
Total Test dengan twin              : 424
